## high-z-accretion-atlas v1
Using functions defined in `src/models.py` and `src/scoring.py`, this code tests `v1_processed.csv` objects against a menu of different seed+growth models and scores feasibility. It also generates core plots for analysis. 


In [ ]:
# imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.models import (
    SEED_MODELS,
    apply_mbh_interpretation,
    apply_mstar_agn_contamination,
    evaluate_seed_model,
    required_average_fedd,
)
from src.scoring import score_model_table
from src.standardize_data import cosmic_time_gyr

In [ ]:
# paths
processed_path = Path('data/processed/v1_processed.csv')
results_dir = Path('results')
results_dir.mkdir(parents=True, exist_ok=True)

# load standardized catalogue
df = pd.read_csv(processed_path)
print(f'Loaded {len(df)} rows from {processed_path}.')
df.head()


Menu of interpretations and seed+growth models (also listed in `docs/model-menu.md`):

In [ ]:
# define interpretation variants (v1 simple branch set)
interpretation_variants = {
    'baseline': {'mbh_delta_dex': 0.0, 'mstar_agn_fraction': 0.0},
    'mbh_minus_0p3dex': {'mbh_delta_dex': -0.3, 'mstar_agn_fraction': 0.0},
    'mbh_plus_0p3dex': {'mbh_delta_dex': 0.3, 'mstar_agn_fraction': 0.0},
    'mstar_agn_20pct': {'mbh_delta_dex': 0.0, 'mstar_agn_fraction': 0.2},
}

# define growth configurations
growth_configs = {
    'eddington_eps0p1': {'f_edd_avg': 1.0, 'epsilon': 0.1, 'merger_boost': 1.0},
    'subeddington_eps0p1': {'f_edd_avg': 0.3, 'epsilon': 0.1, 'merger_boost': 1.0},
    'supercritical_eps0p05': {'f_edd_avg': 2.0, 'epsilon': 0.05, 'merger_boost': 1.0},
    'merger_boost_x2': {'f_edd_avg': 1.0, 'epsilon': 0.1, 'merger_boost': 2.0},
}

# assume seed formation starts at z_seed = 30 for v1 timing budget.
t_seed_gyr = float(cosmic_time_gyr(np.array([30.0]))[0])
print(f'Seed-start cosmic time (z=30): {t_seed_gyr:.4f} Gyr')


In [ ]:
rows = []

for _, obj in df.iterrows():
    delta_t = max(float(obj['cosmic_time_gyr']) - t_seed_gyr, 1e-6)

    for iv_name, iv in interpretation_variants.items():
        log_mbh = float(apply_mbh_interpretation(obj['log_mbh_msun_std'], iv['mbh_delta_dex']))
        log_mstar = float(apply_mstar_agn_contamination(obj['log_mstar_msun_std'], iv['mstar_agn_fraction']))

        for gc_name, gc in growth_configs.items():
            for seed_name, seed_model in SEED_MODELS.items():
                ev = evaluate_seed_model(
                    log_mbh_final=log_mbh,
                    delta_t_gyr=delta_t,
                    model_name=seed_name,
                    f_edd_avg=gc['f_edd_avg'],
                    epsilon=gc['epsilon'],
                    merger_boost=gc['merger_boost'],
                )

                # For required average f_Edd, use geometric-mid prior seed mass.
                log_seed_mid = 0.5 * (seed_model.log_mseed_min + seed_model.log_mseed_max)
                req_fedd = float(
                    required_average_fedd(
                        log_mseed=log_seed_mid,
                        log_mbh_final=log_mbh,
                        delta_t_gyr=delta_t,
                        epsilon=gc['epsilon'],
                        merger_boost=gc['merger_boost'],
                    )
                )

                rows.append({
                    'measurement_id': obj['measurement_id'],
                    'object_id': obj['object_id'],
                    'redshift': float(obj['redshift']),
                    'quality_flag': obj['quality_flag'],
                    'interpretation_variant': iv_name,
                    'growth_config': gc_name,
                    'seed_model': seed_name,
                    'delta_t_gyr': delta_t,
                    'log_mbh_eval': log_mbh,
                    'log_mstar_eval': log_mstar,
                    'log_mbh_mstar_ratio_eval': log_mbh - log_mstar,
                    'required_fedd': req_fedd,
                    **ev,
                })

eval_df = pd.DataFrame(rows)
print(f'Evaluation rows: {len(eval_df)}')
eval_df.head()


In [ ]:
scored_df = score_model_table(eval_df)

requirements_table_path = results_dir / 'v1_requirements_table.csv'
score_table_path = results_dir / 'v1_model_scores.csv'

# Save full evaluation table + scored table
eval_df.to_csv(requirements_table_path, index=False)
scored_df.to_csv(score_table_path, index=False)

print(f'Saved requirements table: {requirements_table_path}')
print(f'Saved score table: {score_table_path}')

scored_df[['measurement_id', 'interpretation_variant', 'growth_config', 'seed_model', 'feasibility_score']].head()


In [ ]:
# Core plot 1: MBH/M* vs z for the standardized catalogue
fig, ax = plt.subplots(figsize=(7, 4.5))
for flag, subset in df.groupby('quality_flag'):
    ax.scatter(
        subset['redshift'],
        subset['log_mbh_mstar_ratio'],
        label=flag,
        alpha=0.8,
    )
ax.set_xlabel('Redshift')
ax.set_ylabel(r'log$_{10}$(M$_{BH}$/M$_*$)')
ax.set_title('v1 catalogue: MBH/M* vs redshift')
ax.legend(title='quality_flag')
ax.grid(alpha=0.2)
fig.tight_layout()
plot1_path = results_dir / 'v1_mbh_mstar_vs_z.png'
fig.savefig(plot1_path, dpi=180)
plt.show()
print(f'Saved: {plot1_path}')


In [ ]:
# Core plot 2: required mean f_Edd vs z (baseline interpretation, eddington_eps0p1)
plot2_df = scored_df[
    (scored_df['interpretation_variant'] == 'baseline')
    & (scored_df['growth_config'] == 'eddington_eps0p1')
].copy()

fig, ax = plt.subplots(figsize=(8, 4.5))
for seed_name, subset in plot2_df.groupby('seed_model'):
    ax.scatter(subset['redshift'], subset['required_fedd'], label=seed_name, alpha=0.75)

ax.axhline(1.0, linestyle='--', linewidth=1.2, color='k', alpha=0.6, label='f_Edd = 1')
ax.set_xlabel('Redshift')
ax.set_ylabel('Required average f_Edd')
ax.set_title('Required average f_Edd vs redshift')
ax.set_yscale('log')
ax.grid(alpha=0.2)
ax.legend(fontsize=8)
fig.tight_layout()
plot2_path = results_dir / 'v1_required_fedd_vs_z.png'
fig.savefig(plot2_path, dpi=180)
plt.show()
print(f'Saved: {plot2_path}')


In [ ]:
# Core plot 3: feasibility heatmap (object x seed model), baseline/eddington only
heat_df = scored_df[
    (scored_df['interpretation_variant'] == 'baseline')
    & (scored_df['growth_config'] == 'eddington_eps0p1')
]

pivot = heat_df.pivot_table(
    index='object_id',
    columns='seed_model',
    values='feasibility_score',
    aggfunc='mean',
).sort_index()

fig, ax = plt.subplots(figsize=(7.5, max(4.0, 0.35 * len(pivot))))
im = ax.imshow(pivot.values, aspect='auto', vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Feasibility score heatmap (baseline, eddington_eps0p1)')
cb = fig.colorbar(im, ax=ax)
cb.set_label('Feasibility score')
fig.tight_layout()
plot3_path = results_dir / 'v1_feasibility_heatmap.png'
fig.savefig(plot3_path, dpi=180)
plt.show()
print(f'Saved: {plot3_path}')


In [ ]:
# Compact summary table for quick inspection
summary = (
    scored_df
    .groupby(['interpretation_variant', 'growth_config', 'seed_model'], as_index=False)['feasibility_score']
    .agg(['mean', 'median', 'min', 'max'])
    .reset_index()
    .sort_values(['mean', 'median'], ascending=False)
)
summary_path = results_dir / 'v1_feasibility_summary.csv'
summary.to_csv(summary_path, index=False)
print(f'Saved: {summary_path}')
summary.head(12)
